In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [2]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_final10.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [5]:
# 데이터를 읽어온다.
train_df = pd.read_csv('train_VIF6.csv')
test_df = pd.read_csv('test_VIF6.csv')

display(train_df)
display(test_df)

,이용금액_R3M_신용체크,소지카드수_이용가능_신용,Life_Stage,_1순위카드이용금액,연령,거주시도명,RV전환가능여부,자발한도감액횟수_R12M,한도증액횟수_R12M,카드론동의여부,...,혜택수혜금액,할인건수_R3M,대표청구지고객주소구분코드,대표결제방법코드,청구금액_R6M,월중평잔_일시불,방문횟수_PC_R6M,이용메뉴건수_ARS_R6M,인입횟수_ARS_R6M,방문횟수_앱_R6M
0,196,1,자녀성장(2),3681,40대,서울,N,0회,0회,Y,...,0,1회 이상,미확인,자동이체,88693,1503,1회 이상,10회 이상,10회 이상,1회 이상
1,13475,1,자녀성장(1),13323,30대,경기,Z,0회,0회,Y,...,0,1회 이상,주거지,자동이체,16861,4447,1회 이상,1회 이상,1회 이상,1회 이상
2,23988,1,자녀출산기,24493,30대,서울,N,0회,0회,Y,...,50,1회 이상,미확인,자동이체,165221,5540,10회 이상,1회 이상,1회 이상,30회 이상
3,3904,2,자녀성장(2),5933,40대,부산,N,0회,0회,Y,...,2,1회 이상,주거지,자동이체,127371,606,1회 이상,10회 이상,10회 이상,1회 이상
4,1190,1,자녀성장(1),0,40대,광주,Z,0회,0회,Y,...,0,1회 이상,주거지,자동이체,155,0,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,10755,1,노년생활,5640,70대이상,울산,Z,0회,0회,Y,...,0,1회 이상,주거지,자동이체,0,0,1회 이상,1회 이상,1회 이상,1회 이상
2399996,27636,1,자녀성장(2),26357,50대,인천,Z,0회,1회이상,Y,...,53,1회 이상,미확인,자동이체,99849,5515,1회 이상,1회 이상,1회 이상,1회 이상
2399997,23187,1,자녀출산기,17171,30대,서울,N,0회,0회,Y,...,0,1회 이상,회사,자동이체,41073,3046,1회 이상,1회 이상,1회 이상,1회 이상
2399998,0,1,자녀성장(1),0,40대,부산,Z,0회,0회,Y,...,0,1회 이상,주거지,자동이체,0,0,1회 이상,1회 이상,1회 이상,1회 이상


,이용금액_R3M_신용체크,소지카드수_이용가능_신용,Life_Stage,_1순위카드이용금액,연령,거주시도명,RV전환가능여부,자발한도감액횟수_R12M,한도증액횟수_R12M,카드론동의여부,...,혜택수혜금액,할인건수_R3M,대표청구지고객주소구분코드,대표결제방법코드,청구금액_R6M,월중평잔_일시불,방문횟수_PC_R6M,이용메뉴건수_ARS_R6M,인입횟수_ARS_R6M,방문횟수_앱_R6M
0,21458,2,자녀성장(1),13852,40대,경기,Z,0회,0회,N,...,56,1회 이상,미확인,자동이체,22151,5187,1회 이상,1회 이상,1회 이상,1회 이상
1,18681,1,자녀독립기,11065,60대,인천,N,0회,0회,Y,...,0,1회 이상,미확인,자동이체,32878,865,1회 이상,1회 이상,1회 이상,1회 이상
2,40758,2,자녀성장(1),27071,40대,경기,Z,0회,1회이상,Y,...,50,1회 이상,미확인,자동이체,71867,5591,1회 이상,1회 이상,1회 이상,1회 이상
3,5255,1,자녀성장(1),4827,40대,인천,Z,0회,0회,Y,...,1,1회 이상,주거지,자동이체,4986,1545,1회 이상,1회 이상,1회 이상,1회 이상
4,16148,1,자녀성장(1),8011,40대,경기,Z,0회,0회,Y,...,0,1회 이상,주거지,자동이체,10758,1462,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,0,0,노년생활,0,60대,경기,NaN,0회,0회,Y,...,0,1회 이상,회사,자동이체,0,0,1회 이상,1회 이상,1회 이상,1회 이상
599996,3110,1,자녀출산기,1231,30대,서울,Z,0회,0회,Y,...,49,1회 이상,주거지,자동이체,2237,256,1회 이상,1회 이상,1회 이상,10회 이상
599997,0,1,자녀성장(1),0,30대,경남,Z,0회,0회,Y,...,0,1회 이상,미확인,자동이체,0,0,1회 이상,1회 이상,1회 이상,1회 이상
599998,173263,3,가족구축기,63592,30대,경남,Z,0회,0회,Y,...,0,1회 이상,미확인,자동이체,108420,14005,1회 이상,1회 이상,1회 이상,40회 이상


In [6]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,이용금액_R3M_신용체크,소지카드수_이용가능_신용,Life_Stage,_1순위카드이용금액,연령,거주시도명,RV전환가능여부,자발한도감액횟수_R12M,한도증액횟수_R12M,카드론동의여부,...,혜택수혜금액,할인건수_R3M,대표청구지고객주소구분코드,대표결제방법코드,청구금액_R6M,월중평잔_일시불,방문횟수_PC_R6M,이용메뉴건수_ARS_R6M,인입횟수_ARS_R6M,방문횟수_앱_R6M
0,196,1,자녀성장(2),3681,40대,서울,N,0회,0회,Y,...,0,1회 이상,미확인,자동이체,88693,1503,1회 이상,10회 이상,10회 이상,1회 이상
1,13475,1,자녀성장(1),13323,30대,경기,Z,0회,0회,Y,...,0,1회 이상,주거지,자동이체,16861,4447,1회 이상,1회 이상,1회 이상,1회 이상
2,23988,1,자녀출산기,24493,30대,서울,N,0회,0회,Y,...,50,1회 이상,미확인,자동이체,165221,5540,10회 이상,1회 이상,1회 이상,30회 이상
3,3904,2,자녀성장(2),5933,40대,부산,N,0회,0회,Y,...,2,1회 이상,주거지,자동이체,127371,606,1회 이상,10회 이상,10회 이상,1회 이상
4,1190,1,자녀성장(1),0,40대,광주,Z,0회,0회,Y,...,0,1회 이상,주거지,자동이체,155,0,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,0,0,노년생활,0,60대,경기,NaN,0회,0회,Y,...,0,1회 이상,회사,자동이체,0,0,1회 이상,1회 이상,1회 이상,1회 이상
2999996,3110,1,자녀출산기,1231,30대,서울,Z,0회,0회,Y,...,49,1회 이상,주거지,자동이체,2237,256,1회 이상,1회 이상,1회 이상,10회 이상
2999997,0,1,자녀성장(1),0,30대,경남,Z,0회,0회,Y,...,0,1회 이상,미확인,자동이체,0,0,1회 이상,1회 이상,1회 이상,1회 이상
2999998,173263,3,가족구축기,63592,30대,경남,Z,0회,0회,Y,...,0,1회 이상,미확인,자동이체,108420,14005,1회 이상,1회 이상,1회 이상,40회 이상


In [7]:
all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 29 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   이용금액_R3M_신용체크   int64 
 1   소지카드수_이용가능_신용   int64 
 2   Life_Stage      object
 3   _1순위카드이용금액      int64 
 4   연령              object
 5   거주시도명           object
 6   RV전환가능여부        object
 7   자발한도감액횟수_R12M   object
 8   한도증액횟수_R12M     object
 9   카드론동의여부         object
 10  한도심사요청건수        object
 11  카드이용한도금액_B2M    int64 
 12  최대이용금액_할부_R12M  int64 
 13  이용금액_일시불_R12M   int64 
 14  _3순위업종_이용금액     int64 
 15  이용금액_오프라인_B0M   int64 
 16  연체입금원금_B0M      int64 
 17  이용금액대           object
 18  이용후경과월_할부_무이자   int64 
 19  혜택수혜금액          int64 
 20  할인건수_R3M        object
 21  대표청구지고객주소구분코드   object
 22  대표결제방법코드        object
 23  청구금액_R6M        int64 
 24  월중평잔_일시불        int64 
 25  방문횟수_PC_R6M     object
 26  이용메뉴건수_ARS_R6M  object
 27  인입횟수_ARS_R6M    object
 28  방문횟수_앱_R6M      object
dtypes: int64(13), 

In [8]:
# LabelEncoder 학습
Encoder1 = LabelEncoder()
Encoder2 = LabelEncoder()
Encoder3 = LabelEncoder()
Encoder4 = LabelEncoder()
Encoder5 = LabelEncoder()
Encoder6 = LabelEncoder()
Encoder7 = LabelEncoder()
Encoder8 = LabelEncoder()
Encoder9 = LabelEncoder()
Encoder10 = LabelEncoder()
Encoder11 = LabelEncoder()
Encoder12 = LabelEncoder()
Encoder13 = LabelEncoder()
Encoder14 = LabelEncoder()
Encoder15 = LabelEncoder()
Encoder16 = LabelEncoder()

Encoder1.fit(all_df['방문횟수_앱_R6M'])
Encoder2.fit(all_df['RV전환가능여부'])
Encoder3.fit(all_df['자발한도감액횟수_R12M'])
Encoder4.fit(all_df['한도증액횟수_R12M'])
Encoder5.fit(all_df['카드론동의여부'])
Encoder6.fit(all_df['한도심사요청건수'])
Encoder7.fit(all_df['이용금액대'])
Encoder8.fit(all_df['할인건수_R3M'])
Encoder9.fit(all_df['대표결제방법코드'])
Encoder10.fit(all_df['방문횟수_PC_R6M'])
Encoder11.fit(all_df['이용메뉴건수_ARS_R6M'])
Encoder12.fit(all_df['인입횟수_ARS_R6M'])
Encoder13.fit(all_df['대표청구지고객주소구분코드'])
Encoder14.fit(all_df['Life_Stage'])
Encoder15.fit(all_df['거주시도명'])
Encoder16.fit(all_df['연령'])

LabelEncoder()

In [9]:
all_df['방문횟수_앱_R6M'] = Encoder1.transform(all_df['방문횟수_앱_R6M'])
all_df['RV전환가능여부'] = Encoder2.transform(all_df['RV전환가능여부'])
all_df['자발한도감액횟수_R12M'] = Encoder3.transform(all_df['자발한도감액횟수_R12M'])
all_df['한도증액횟수_R12M'] = Encoder4.transform(all_df['한도증액횟수_R12M'])
all_df['카드론동의여부'] = Encoder5.transform(all_df['카드론동의여부'])
all_df['한도심사요청건수'] = Encoder6.transform(all_df['한도심사요청건수'])
all_df['이용금액대'] = Encoder7.transform(all_df['이용금액대'])
all_df['할인건수_R3M'] = Encoder8.transform(all_df['할인건수_R3M'])
all_df['대표결제방법코드'] = Encoder9.transform(all_df['대표결제방법코드'])
all_df['방문횟수_PC_R6M'] = Encoder10.transform(all_df['방문횟수_PC_R6M'])
all_df['이용메뉴건수_ARS_R6M'] = Encoder11.transform(all_df['이용메뉴건수_ARS_R6M'])
all_df['인입횟수_ARS_R6M'] = Encoder12.transform(all_df['인입횟수_ARS_R6M'])
all_df['대표청구지고객주소구분코드'] = Encoder13.transform(all_df['대표청구지고객주소구분코드'])
all_df['Life_Stage'] = Encoder14.transform(all_df['Life_Stage'])
all_df['거주시도명'] = Encoder15.transform(all_df['거주시도명'])
all_df['연령'] = Encoder16.transform(all_df['연령'])

In [10]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

,copy,True
,with_mean,True
,with_std,True


In [11]:
train_df['방문횟수_앱_R6M'] = Encoder1.transform(train_df['방문횟수_앱_R6M'])
train_df['RV전환가능여부'] = Encoder2.transform(train_df['RV전환가능여부'])
train_df['자발한도감액횟수_R12M'] = Encoder3.transform(train_df['자발한도감액횟수_R12M'])
train_df['한도증액횟수_R12M'] = Encoder4.transform(train_df['한도증액횟수_R12M'])
train_df['카드론동의여부'] = Encoder5.transform(train_df['카드론동의여부'])
train_df['한도심사요청건수'] = Encoder6.transform(train_df['한도심사요청건수'])
train_df['이용금액대'] = Encoder7.transform(train_df['이용금액대'])
train_df['할인건수_R3M'] = Encoder8.transform(train_df['할인건수_R3M'])
train_df['대표결제방법코드'] = Encoder9.transform(train_df['대표결제방법코드'])
train_df['방문횟수_PC_R6M'] = Encoder10.transform(train_df['방문횟수_PC_R6M'])
train_df['이용메뉴건수_ARS_R6M'] = Encoder11.transform(train_df['이용메뉴건수_ARS_R6M'])
train_df['인입횟수_ARS_R6M'] = Encoder12.transform(train_df['인입횟수_ARS_R6M'])
train_df['대표청구지고객주소구분코드'] = Encoder13.transform(train_df['대표청구지고객주소구분코드'])
train_df['Life_Stage'] = Encoder14.transform(train_df['Life_Stage'])
train_df['거주시도명'] = Encoder15.transform(train_df['거주시도명'])
train_df['연령'] = Encoder16.transform(train_df['연령'])

In [12]:
target1=pd.read_parquet(r'data/train/1.회원정보/201807_train_.parquet')
target2=pd.read_parquet(r'data/train/1.회원정보/201808_train_.parquet')
target3=pd.read_parquet(r'data/train/1.회원정보/201809_train_.parquet')
target4=pd.read_parquet(r'data/train/1.회원정보/201810_train_.parquet')
target5=pd.read_parquet(r'data/train/1.회원정보/201811_train_.parquet')
target6=pd.read_parquet(r'data/train/1.회원정보/201812_train_.parquet')

In [13]:
tg_df = pd.concat([
    target1['Segment'],
    target2['Segment'],
    target3['Segment'],
    target4['Segment'],
    target5['Segment'],
    target6['Segment']
])

tg_df = tg_df.reset_index(drop=True).to_frame(name='Segment')
tg_df

,Segment
0,D
1,E
2,C
3,D
4,E
...,...
2399995,E
2399996,D
2399997,C
2399998,E


In [14]:
# 라벨 인코더 생성
le = LabelEncoder()

# 문자열 y를 숫자로 변환
tg_df['Segment'] = le.fit_transform(tg_df['Segment'])

In [15]:
# 입력과 결과로 나눈다.
X = train_df
y = tg_df

In [16]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-0.71919385, -0.24605525,  0.78248617, ..., -4.44973122,
        -5.69855686, -0.22479894],
       [-0.14848993, -0.24605525,  0.22559285, ...,  0.08322662,
         0.17548303, -0.22479894],
       [ 0.30333704, -0.24605525,  1.3393795 , ...,  0.08322662,
         0.17548303,  1.94897776],
       ...,
       [ 0.26891172, -0.24605525,  1.3393795 , ...,  0.08322662,
         0.17548303, -0.22479894],
       [-0.72761753, -0.24605525,  0.22559285, ...,  0.08322662,
         0.17548303, -0.22479894],
       [ 0.19481777, -0.24605525,  0.22559285, ...,  0.08322662,
         0.17548303, -0.22479894]])

In [17]:
train_X = X2
train_y = y

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [16]:
model5 = LGBMClassifier(device='cpu', objective='multiclass', num_class=5, verbose=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
r1 = cross_val_score(model5, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r1.mean()}')

f1_score_list.append(r1.mean())
model_name_list.append("LGBMClassifier")

KeyboardInterrupt: 

In [18]:
# CPU 기반 XGBoost 모델
model6 = XGBClassifier(
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# 교차 검증
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

# f1_weighted 사용
r2 = cross_val_score(model6, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r2.mean():.4f}')

f1_score_list.append(r2.mean())
model_name_list.append("XGBClassifier")

평균 f1 Score : 0.8786


In [20]:
final_model=model6.fit(train_X, train_y)

In [21]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(model6, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(Encoder1, fp)
    pickle.dump(Encoder2, fp)
    pickle.dump(Encoder3, fp)
    pickle.dump(Encoder4, fp)
    pickle.dump(Encoder5, fp)
    pickle.dump(Encoder6, fp)
    pickle.dump(Encoder7, fp)
    pickle.dump(Encoder8, fp)
    pickle.dump(Encoder9, fp)
    pickle.dump(Encoder10, fp)
    pickle.dump(Encoder11, fp)
    pickle.dump(Encoder12, fp)
    pickle.dump(Encoder13, fp)
    pickle.dump(Encoder14, fp)
    pickle.dump(Encoder15, fp)
    pickle.dump(Encoder16, fp)
    pickle.dump(le, fp)  # ← LabelEncoder 객체 추가

print('저장완료')

저장완료
